# P2C-64 v1.1 (SYGMA-SCONE) — Google Colab A100 40GB 実行ノートブック（マイルストーン2B）

このノートブックは、Egaroucid 公開学習データ（`Egaroucid_Train_Data.zip`, 約2,551万局面）を用いて、次世代純スカラーオセロ評価関数 **P2C-64 v1.1** の全5モデル（`RAW_NTUPLE_64`, `LINEAR_64`, `CAND_A_SCRELU_K2`, `CAND_B_3W3B`, `CAND_C_HYBRID`）を **上から実行するだけで完走** させるための本番パイプラインです。

### 前提パス（Google Drive）
- 入力ZIP: `/content/drive/MyDrive/p2c64_data/Egaroucid_Train_Data.zip`
- 変換済みキャッシュ: `/content/drive/MyDrive/p2c64_data/cache_bin/{train.bin, val.bin}`
- 学習出力・チェックポイント: `/content/drive/MyDrive/p2c64_runs/<RUN_NAME>/`

> **運用Tips**: セル 1〜6 は **A100 GPU ランタイム** で実行し、全 `.p2c` の保存とビット完全一致監査が終わったら **CPUランタイムへ切替**（CU節約）してからセル 7（純スカラー `match_engine` 1,000局対局 ＆ MPC $\sigma$ 計測）を実行してください。

## Cell 1: GPU/CPU 環境確認 ＆ Googleドライブマウント

In [ ]:
import os
import torch
from google.colab import drive

drive.mount("/content/drive")

RUN_NAME = "run_egaroucid_25m_v11"
DRIVE_DATA_DIR = "/content/drive/MyDrive/p2c64_data"
DRIVE_ZIP_PATH = f"{DRIVE_DATA_DIR}/Egaroucid_Train_Data.zip"
DRIVE_CACHE_DIR = f"{DRIVE_DATA_DIR}/cache_bin"
DRIVE_WORK_DIR = f"/content/drive/MyDrive/p2c64_runs/{RUN_NAME}"
LOCAL_DATA_DIR = "/content/data"
REPO_DIR = "/content/p2c64-othello"

os.makedirs(DRIVE_CACHE_DIR, exist_ok=True)
os.makedirs(DRIVE_WORK_DIR, exist_ok=True)
os.makedirs(LOCAL_DATA_DIR, exist_ok=True)

print("PyTorch :", torch.__version__, "| CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU     :", torch.cuda.get_device_name(0))
    !nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv
!lscpu | grep -E "Model name|CPU\(s\):|L1d|L2|L3" | head -n 10

## Cell 2: リポジトリ `clone` / `pull` ＆ C++ツール・OpenMPバッチブリッジビルド

In [ ]:
%%bash
set -euo pipefail
REPO_URL="https://github.com/Taichi111009/p2c64-othello.git"
REPO_DIR="/content/p2c64-othello"

if [ -d "$REPO_DIR/.git" ]; then
    cd "$REPO_DIR" && git fetch origin && git pull --ff-only origin main
else
    git clone "$REPO_URL" "$REPO_DIR"
fi

cd "$REPO_DIR"
make clean all test audit
ls -lh build/

## Cell 3: スマートデータ準備（Driveキャッシュ再利用 / 初回のみ高速ローカルSSD変換）

In [ ]:
import os
import shutil
import subprocess

cache_tr = f"{DRIVE_CACHE_DIR}/train.bin"
cache_va = f"{DRIVE_CACHE_DIR}/val.bin"
local_tr = f"{LOCAL_DATA_DIR}/train.bin"
local_va = f"{LOCAL_DATA_DIR}/val.bin"

if os.path.exists(local_tr) and os.path.exists(local_va):
    print("[OK] Local SSD binary cache already present:", local_tr, local_va)
elif os.path.exists(cache_tr) and os.path.exists(cache_va):
    print("[Cache Hit] Copying pre-converted train.bin / val.bin from Google Drive -> Local SSD...")
    shutil.copy2(cache_tr, local_tr)
    shutil.copy2(cache_va, local_va)
    print("[OK] Copied to local SSD!")
else:
    assert os.path.exists(DRIVE_ZIP_PATH), f"Input ZIP not found: {DRIVE_ZIP_PATH}"
    local_zip = "/content/Egaroucid_Train_Data.zip"
    unzip_dir = "/content/egaroucid_raw"
    print(f"[Cache Miss] Copying {DRIVE_ZIP_PATH} -> {local_zip} ...")
    shutil.copy2(DRIVE_ZIP_PATH, local_zip)
    os.makedirs(unzip_dir, exist_ok=True)
    print("Unzipping on local SSD...")
    subprocess.run(["unzip", "-q", "-o", local_zip, "-d", unzip_dir], check=True)

    print("Running C++ preprocess_egaroucid (leak-free D4xZ2 canonical 95/5 split)...")
    subprocess.run([
        f"{REPO_DIR}/build/preprocess_egaroucid",
        "--input-dir", unzip_dir,
        "--out-train", local_tr,
        "--out-val", local_va,
        "--val-pct", "5",
    ], check=True)

    print("Saving binary cache back to Google Drive for future sessions...")
    shutil.copy2(local_tr, cache_tr)
    shutil.copy2(local_va, cache_va)
    # Free local SSD space
    os.remove(local_zip)
    shutil.rmtree(unzip_dir, ignore_errors=True)
    print("[OK] Data preparation & Drive caching complete!")

!ls -lh /content/data/

## Cell 4: Phase 1 学習（`RAW_NTUPLE_64` ＆ `LINEAR_64`）＋ `.p2c` 自動保存（中断再開対応）

In [ ]:
!python3 {REPO_DIR}/python/train_p2c64.py \
    --train-bin {LOCAL_DATA_DIR}/train.bin \
    --val-bin {LOCAL_DATA_DIR}/val.bin \
    --work-dir {DRIVE_WORK_DIR} \
    --phases phase1 \
    --resume \
    --batch-size 32768 \
    --val-batch-size 65536 \
    --p1-epochs 6 \
    --p1-lr 0.05 \
    --smooth-lambda 1e-4 \
    --huber-beta 4.0

## Cell 5: Phase 2 学習（候補A・候補B・候補C）＋ `.p2c` 自動保存 ＋ Val RMSE 比較サマリー

In [ ]:
!python3 {REPO_DIR}/python/train_p2c64.py \
    --train-bin {LOCAL_DATA_DIR}/train.bin \
    --val-bin {LOCAL_DATA_DIR}/val.bin \
    --work-dir {DRIVE_WORK_DIR} \
    --phases phase2 \
    --resume \
    --batch-size 16384 \
    --val-batch-size 65536 \
    --a-epochs 4 --a-qat-epochs 2 --a-lr 0.01 --mu-init 1.0,0.35 \
    --b-soft-epochs 4 --b-lr 0.02 --tau-start 1.5 --tau-end 0.15 --eb-lambda 200.0 --eb-amax 4.0 --em-iters 2 \
    --c-screlu-epochs 3 --c-qat-epochs 1 --c-soft-epochs 3 --c-em-iters 2

print("\nSaved artifacts on Google Drive:")
!ls -lh {DRIVE_WORK_DIR}/*.p2c {DRIVE_WORK_DIR}/metrics.*

## Cell 6: C++/Python ビット完全一致監査 ＆ 16対称（$D_4 \times \mathbb{Z}_2$）テスト

In [ ]:
!python3 {REPO_DIR}/python/test_bit_exact_io.py \
    --data {LOCAL_DATA_DIR}/val.bin \
    --num-pos 1000 \
    --work-dir {DRIVE_WORK_DIR} \
    --bench-bridge

## Cell 7: （GPU切断後のCPUランタイム推奨）純スカラー実戦1,000局ペア対局 ＆ MPC $\sigma$ 計測

Colabのランタイムを **CPU（ハイメモリ不要）** へ切り替えてから、Cell 1〜2（ドライブマウント＆ビルド）を再実行し、以下のセルで実戦対局・MPC $\sigma$ 計測を実行してください。結果は Google Drive の `DRIVE_WORK_DIR` に CSV として保存されます。

In [ ]:
import os
import subprocess

MATCH_BIN = f"{REPO_DIR}/build/match_engine"
MPC_BIN = f"{REPO_DIR}/build/measure_mpc_sigma"
RAW_P2C = f"{DRIVE_WORK_DIR}/raw_ntuple64.p2c"
LIN_P2C = f"{DRIVE_WORK_DIR}/linear64.p2c"
CANDA_P2C = f"{DRIVE_WORK_DIR}/cand_a_screlu_k2.p2c"
CANDB_P2C = f"{DRIVE_WORK_DIR}/cand_b_3w3b.p2c"
CANDC_P2C = f"{DRIVE_WORK_DIR}/cand_c_hybrid.p2c"

val_bin = f"{LOCAL_DATA_DIR}/val.bin" if os.path.exists(f"{LOCAL_DATA_DIR}/val.bin") else f"{DRIVE_CACHE_DIR}/val.bin"

# 1) MPC sigma measurement (depth 2 -> 6 and depth 4 -> 8 on 2,000 unseen positions)
for sh, dp in [(2, 6), (4, 8)]:
    print(f"\n=== MPC Sigma: depth {sh} -> {dp} ===")
    subprocess.run([
        MPC_BIN,
        "--data", val_bin,
        "--num-pos", "2000",
        "--shallow", str(sh),
        "--deep", str(dp),
        "--tt-mb", "64",
        "--csv", f"{DRIVE_WORK_DIR}/mpc_sigma_d{sh}_d{dp}.csv",
        "--models", RAW_P2C, LIN_P2C, CANDA_P2C, CANDB_P2C, CANDC_P2C,
    ], check=True)

# 2) 1,000-game paired matches (500 pairs): Equal-Nodes & Equal-Time vs RAW_NTUPLE_64
match_csv = f"{DRIVE_WORK_DIR}/match_results.csv"
for cand_path in [LIN_P2C, CANDA_P2C, CANDB_P2C, CANDC_P2C]:
    for mode, limit in [("nodes", "20000"), ("time", "20")]:
        print(f"\n=== 1,000-Game Match ({mode}={limit}): {os.path.basename(cand_path)} vs RAW_NTUPLE_64 ===")
        subprocess.run([
            MATCH_BIN,
            "--p1", cand_path,
            "--p2", RAW_P2C,
            "--mode", mode,
            "--limit", limit,
            "--pairs", "500",
            "--opening-plies", "8",
            "--endgame-empties", "10",
            "--tt-mb", "128",
            "--seed", "20261125",
            "--csv", match_csv,
        ], check=True)

print(f"\nAll match & MPC results saved to {DRIVE_WORK_DIR}")